In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as T
from torch.amp import autocast, GradScaler
from torch.utils.tensorboard import SummaryWriter
# 1) Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

writer = SummaryWriter(log_dir="runs/resnext29_cifar10")

Using device: cuda


In [3]:
transform_train = T.Compose([
    T.RandomHorizontalFlip(),
    T.RandomCrop(32, padding=4),
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = T.Compose([
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
testset  = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

train_loader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True, num_workers=4)
test_loader  = torch.utils.data.DataLoader(testset,  batch_size=128,  shuffle=False, num_workers=4)

In [4]:
from resnext import resnext29_8x64d  # assuming you defined this from previous step

model = resnext29_8x64d(num_classes=10).to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[150, 225], gamma=0.1)
epochs = 300
batch_size = 512
criterion = nn.CrossEntropyLoss().to(device)
scaler = GradScaler(init_scale=2**12, device="cuda")


In [5]:
for epoch in range(1, epochs + 1):
    model.train()
    total_loss, correct = 0.0, 0
    total = 0

    for batch_idx, (inputs, targets) in enumerate(train_loader):
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()

        with autocast(device_type=device.type):
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

    scheduler.step()

    train_acc = 100. * correct / total
    train_loss = total_loss / total

    # Validation
    model.eval()
    val_loss = val_correct = val_total = 0
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)

            val_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            val_total += targets.size(0)
            val_correct += predicted.eq(targets).sum().item()

    val_acc = 100. * val_correct / val_total
    val_loss = val_loss / val_total

    writer.add_scalar("LR/epoch", optimizer.param_groups[0]["lr"], epoch)
    writer.add_scalars("Accuracy", {"Train": train_acc, "Validation": val_acc}, epoch)
    writer.add_scalars("Loss",     {"Train": train_loss, "Validation": val_loss}, epoch)

    print(f"Epoch {epoch:03d}: Train Loss {train_loss:.4f}, Acc {train_acc:.2f}% | Val Loss {val_loss:.4f}, Acc {val_acc:.2f}%")

Epoch 001: Train Loss 2.6799, Acc 16.04% | Val Loss 2.0856, Acc 22.64%
Epoch 002: Train Loss 1.8372, Acc 29.56% | Val Loss 1.7415, Acc 34.41%
Epoch 003: Train Loss 1.5793, Acc 41.52% | Val Loss 1.4670, Acc 45.74%
Epoch 004: Train Loss 1.3829, Acc 49.84% | Val Loss 1.4764, Acc 48.12%
Epoch 005: Train Loss 1.1675, Acc 58.24% | Val Loss 1.2575, Acc 56.38%
Epoch 006: Train Loss 1.0005, Acc 64.61% | Val Loss 1.0270, Acc 63.36%
Epoch 007: Train Loss 0.8458, Acc 70.27% | Val Loss 0.8713, Acc 69.86%
Epoch 008: Train Loss 0.7261, Acc 74.78% | Val Loss 0.9562, Acc 67.10%
Epoch 009: Train Loss 0.6421, Acc 77.78% | Val Loss 0.8928, Acc 70.05%
Epoch 010: Train Loss 0.5929, Acc 79.62% | Val Loss 0.9849, Acc 67.35%
Epoch 011: Train Loss 0.5565, Acc 80.75% | Val Loss 0.6676, Acc 77.61%
Epoch 012: Train Loss 0.5238, Acc 81.94% | Val Loss 0.6972, Acc 76.78%
Epoch 013: Train Loss 0.5063, Acc 82.59% | Val Loss 0.7473, Acc 74.95%
Epoch 014: Train Loss 0.4960, Acc 82.88% | Val Loss 0.5991, Acc 79.83%
Epoch 

In [6]:
def evaluate(model, dataloader, device):
    model.eval()
    correct = total = 0
    loss_sum = 0.0
    criterion = nn.CrossEntropyLoss()

    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)

            loss_sum += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

    avg_loss = loss_sum / total
    accuracy = 100. * correct / total
    return avg_loss, accuracy

In [7]:
test_loss, test_acc = evaluate(model, test_loader, device)
print(f"Final Test Loss: {test_loss:.4f} | Final Test Accuracy: {test_acc:.2f}%")

Final Test Loss: 0.1332 | Final Test Accuracy: 95.88%


In [8]:
save_path = "resnext29_cifar10.pth"
torch.save(model.state_dict(), save_path)
print(f"Model weights saved to {save_path}")
sd = torch.load(save_path, map_location="cpu")
print(sd["fc.weight"].shape, sd["fc.bias"].shape)

Model weights saved to resnext29_cifar10.pth
torch.Size([10, 1024]) torch.Size([10])
